# CIFAR ResNet epoch 144 latent clustering

This notebook loads latent vectors (`a3`, i.e., layer 3) for `resnet_cifar` at epoch 144 from multifield inference outputs in `~/data_2` (fallback: `~/data_1`), runs multiple clustering algorithms, and visualizes:
- class-vs-cluster heatmaps
- dendrograms for hierarchical structure

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

from scipy.cluster.hierarchy import linkage, dendrogram

sns.set_theme(style="whitegrid")

In [ ]:
MODEL_DATASET = "resnet_cifar"
SPLIT = "trainUval"
EPOCH = 144
TENSOR_TAG = 3  # a3 corresponds to layer 3 for this run config
SEED = 42

CANDIDATE_ROOTS = [
    Path("~/data_2/tvcg_multifield_infer").expanduser(),
    Path("~/data_1/tvcg_multifield_infer").expanduser(),
]

In [ ]:
def resolve_run_dir(model_dataset: str, candidates: list[Path]) -> Path:
    for root in candidates:
        run_dir = root / model_dataset
        if run_dir.exists():
            return run_dir
    checked = "\n".join(str(p / model_dataset) for p in candidates)
    raise FileNotFoundError(
        f"Could not find multifield run directory for {model_dataset}. Checked:\n{checked}"
    )


def load_epoch_latents_and_labels(
    run_dir: Path, split: str, epoch: int, tensor_tag: int
) -> tuple[np.ndarray, np.ndarray]:
    tensor_path = run_dir / "Tensors" / split / f"a{tensor_tag}_e{epoch}.pt"
    labels_path = run_dir / "Labels" / split / f"labels_e{epoch}.pt"

    if not tensor_path.exists():
        raise FileNotFoundError(f"Missing latent tensor file: {tensor_path}")
    if not labels_path.exists():
        raise FileNotFoundError(f"Missing labels file: {labels_path}")

    latents = torch.load(tensor_path, map_location="cpu")
    labels = torch.load(labels_path, map_location="cpu")

    if isinstance(latents, torch.Tensor):
        latents = latents.detach().cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.detach().cpu().numpy()

    latents = np.asarray(latents)
    labels = np.asarray(labels).astype(int)

    if latents.ndim != 2:
        raise ValueError(f"Expected 2D latent matrix, got shape {latents.shape}")
    if labels.ndim != 1:
        labels = labels.reshape(-1)
    if latents.shape[0] != labels.shape[0]:
        raise ValueError(
            f"Latent/label length mismatch: {latents.shape[0]} vs {labels.shape[0]}"
        )

    return latents, labels

In [ ]:
run_dir = resolve_run_dir(MODEL_DATASET, CANDIDATE_ROOTS)
X_raw, y = load_epoch_latents_and_labels(run_dir, SPLIT, EPOCH, TENSOR_TAG)

print(f"Using run directory: {run_dir}")
print(f"Latent matrix shape: {X_raw.shape}")
print(f"Labels shape: {y.shape}")
print("Class counts:")
print(pd.Series(y).value_counts().sort_index())

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

max_dims = min(64, X_scaled.shape[1], X_scaled.shape[0] - 1)
if max_dims < 2:
    raise ValueError(f"Not enough samples/features for PCA: {X_scaled.shape}")

pca = PCA(n_components=max_dims, random_state=SEED)
X = pca.fit_transform(X_scaled)

print(f"Reduced latent shape: {X.shape}")
print(f"Explained variance ratio sum: {pca.explained_variance_ratio_.sum():.4f}")

In [ ]:
n_classes = int(np.unique(y).shape[0])

clusterers = {
    "KMeans": KMeans(n_clusters=n_classes, random_state=SEED, n_init=20),
    "GaussianMixture": GaussianMixture(n_components=n_classes, random_state=SEED),
    "Agglomerative (ward)": AgglomerativeClustering(n_clusters=n_classes, linkage="ward"),
    "Agglomerative (average-cosine)": AgglomerativeClustering(
        n_clusters=n_classes,
        linkage="average",
        metric="cosine",
    ),
}

cluster_assignments = {}
scores = []

for name, estimator in clusterers.items():
    if isinstance(estimator, GaussianMixture):
        labels_pred = estimator.fit_predict(X)
    else:
        labels_pred = estimator.fit_predict(X)

    cluster_assignments[name] = labels_pred
    scores.append(
        {
            "algorithm": name,
            "ARI": adjusted_rand_score(y, labels_pred),
            "NMI": normalized_mutual_info_score(y, labels_pred),
            "n_clusters_found": int(np.unique(labels_pred).shape[0]),
        }
    )

scores_df = pd.DataFrame(scores).sort_values("NMI", ascending=False)
scores_df

In [ ]:
def plot_class_cluster_heatmap(true_labels: np.ndarray, cluster_labels: np.ndarray, title: str) -> None:
    table = pd.crosstab(
        pd.Series(true_labels, name="class"),
        pd.Series(cluster_labels, name="cluster"),
        normalize="index",
    )

    plt.figure(figsize=(12, 5))
    sns.heatmap(table, cmap="mako", vmin=0.0, vmax=1.0)
    plt.title(title)
    plt.xlabel("cluster id")
    plt.ylabel("true class")
    plt.tight_layout()
    plt.show()

In [ ]:
for name, pred in cluster_assignments.items():
    plot_class_cluster_heatmap(y, pred, f"{name}: class vs cluster (row-normalized)")

In [ ]:
# Dendrogram over class centroids in latent space.
classes = np.sort(np.unique(y))
class_centroids = np.vstack([X[y == c].mean(axis=0) for c in classes])

Z = linkage(class_centroids, method="ward", metric="euclidean")

plt.figure(figsize=(12, 5))
dendrogram(Z, labels=[f"class {c}" for c in classes], leaf_rotation=45)
plt.title("Hierarchical dendrogram of class centroids (ward linkage)")
plt.xlabel("class")
plt.ylabel("distance")
plt.tight_layout()
plt.show()

In [ ]:
# Optional: sample-level dendrogram for a manageable random subset.
rng = np.random.default_rng(SEED)
sample_size = min(400, X.shape[0])
sample_idx = rng.choice(X.shape[0], size=sample_size, replace=False)
X_sample = X[sample_idx]
y_sample = y[sample_idx]

Z_sample = linkage(X_sample, method="ward", metric="euclidean")

plt.figure(figsize=(16, 6))
dendrogram(
    Z_sample,
    labels=[str(cls) for cls in y_sample],
    leaf_rotation=90,
    leaf_font_size=7,
)
plt.title(f"Sample-level dendrogram (n={sample_size}, leaf label=true class)")
plt.xlabel("sample")
plt.ylabel("distance")
plt.tight_layout()
plt.show()